# Camada analítica — BCI IV 2a (Parquet + DuckDB)

Esquema estrela sobre o 2a: uma tabela fato **trial-a-trial** e dimensões (sujeito, classe,
sessão, run), em Parquet e consultadas com DuckDB. O desenho do esquema está documentado no
[`../README.md`](../README.md). A ingestão vive em [`../src/ingest.py`](../src/ingest.py)
(constrói as tabelas a partir dos metadados do MOABB, **sem** o sinal bruto).

## Tabelas

As tabelas são geradas por `python -m src.ingest`. A célula abaixo as regera apenas se ainda
não existirem (evita rebaixar o dataset a cada execução).

In [ ]:
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, "..")
P = Path("..") / "data" / "processed"

if not (P / "fact_trial.parquet").exists():
    from src.ingest import build_tables, write_tables
    write_tables(build_tables(), P)
    print("tabelas geradas")
else:
    print("tabelas presentes em", P)

con = duckdb.connect()
fact = f"'{P.as_posix()}/fact_trial.parquet'"
dim_class = f"'{P.as_posix()}/dim_class.parquet'"
dim_session = f"'{P.as_posix()}/dim_session.parquet'"
dim_subject = f"'{P.as_posix()}/dim_subject.parquet'"

## Schema explícito e nulos verdadeiros

Identificadores como texto, tipos definidos, e demografia ausente como **NULL** (não `-1`/`999`).

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM {fact}").df()

In [ ]:
# dim_subject: age/sex/handedness são NULL (o 2a não publica demografia)
con.sql(f"SELECT * FROM {dim_subject}").df()

## Consultas (respondem o EDA do 2a)

In [ ]:
# Q1 — as 4 classes estão balanceadas em cada papel de sessão?
con.sql(f"""
    SELECT s.session_role, c.class_name, COUNT(*) AS n_trials
    FROM {fact} f
    JOIN {dim_class} c USING (class_id)
    JOIN {dim_session} s USING (session_id)
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

In [ ]:
# Q2 — quantos trials por papel de sessão (treino vs teste)?
con.sql(f"""
    SELECT s.session_role, COUNT(*) AS n_trials
    FROM {fact} f
    JOIN {dim_session} s USING (session_id)
    GROUP BY 1
""").df()

In [ ]:
# Q3 — cobertura: trials por sujeito
con.sql(f"""
    SELECT subject_id, COUNT(*) AS n_trials
    FROM {fact}
    GROUP BY 1
    ORDER BY 1
""").df()

## Benchmark — CSV vs Parquet

Mesma tabela fato nos dois formatos: tamanho em disco e tempo médio de uma agregação.

In [ ]:
import os, time

csv_path = (P / "fact_trial.csv").as_posix()
pq_path = (P / "fact_trial.parquet").as_posix()

csv_kib = os.path.getsize(csv_path) / 1024
pq_kib = os.path.getsize(pq_path) / 1024
print(f"CSV:     {csv_kib:7.1f} KiB")
print(f"Parquet: {pq_kib:7.1f} KiB  ({csv_kib / pq_kib:.1f}x menor)")

def bench(query, n=20):
    t = time.perf_counter()
    for _ in range(n):
        con.sql(query).fetchall()
    return (time.perf_counter() - t) / n * 1000

ms_csv = bench(f"SELECT class_id, COUNT(*) FROM read_csv_auto('{csv_path}') GROUP BY 1")
ms_pq = bench(f"SELECT class_id, COUNT(*) FROM read_parquet('{pq_path}') GROUP BY 1")
print(f"\nConsulta CSV:     {ms_csv:6.2f} ms")
print(f"Consulta Parquet: {ms_pq:6.2f} ms  ({ms_csv / ms_pq:.0f}x mais rápido)")